In [ ]:
# ============================================================
# ASSIGNMENT 2 - Q3.4: BASELINE -> BEATEN, with PAIRED BOOTSTRAP 95% CIs (MIND)
# Team: Shivam Patel + Suyash Pande | CS4.406 IRE
#
# Q3 asks: reproduce a baseline, beat it with a principled change, ablate the improvement,
# and ship a PAIRED bootstrap 95% CI on the gain that EXCLUDES ZERO.
#
# Baseline: NRMS title-only (dev AUC ~0.585).
# Improvements (Option C, two ways):
#   1. Behavioural features (GBDT / MLP) - the feature-model improvement
#   2. Richer text (NRMS title+abstract) - the neural improvement
# Ablations (documented): NRMS +category (null), +recency (regression),
#   and content-only-vs-full GBDT feature ablation.
#
# Paired unit = per-impression AUC on the SHARED dev impressions. For each pair
# (improvement vs baseline) we bootstrap the mean per-impression AUC difference and
# report the 95% CI; excluding zero => the improvement is significant.
#
# Loads dev-score caches (.pkl) + features npz from HF (public, no token).
# NEEDS: Kaggle MIND-large mounted (for labels/impressions alignment).
# ============================================================
!pip install lightgbm huggingface_hub -q
import glob, numpy as np, pandas as pd, pickle
from huggingface_hub import hf_hub_download
HF_REPO="donbosoc/mind-artifacts"
def dl(name): return hf_hub_download(HF_REPO, name, repo_type="dataset")
LDV=glob.glob("/kaggle/input/**/MINDlarge_dev",recursive=True)[0]
BEH=["impression_id","user_id","time","history","impressions"]
def pfx(x): return f"mind:{x}"

# ---- load NRMS dev-score caches (keyed by (impression_id, article_id)) ----
base   = pickle.load(open(dl("mind_nrms_devscores.pkl"),"rb"))       # baseline (title-only)
ta     = pickle.load(open(dl("mind_nrms_ta_devscores.pkl"),"rb"))    # +abstract
cat    = pickle.load(open(dl("mind_nrms_cat_devscores.pkl"),"rb"))   # +category
fresh  = pickle.load(open(dl("mind_nrms_fresh_devscores.pkl"),"rb")) # +recency
print("caches loaded:",
      "base",len(base),"ta",len(ta),"cat",len(cat),"fresh",len(fresh))


In [ ]:
# ---- reconstruct labels for the dev impressions that the caches cover ----
b_dv=pd.read_csv(f"{LDV}/behaviors.tsv",sep="\t",header=None,names=BEH,quoting=3)
# map impression_id -> list of (article_id, label)
imp_slate={}
for iid,imps in zip(b_dv["impression_id"],b_dv["impressions"]):
    if not isinstance(imps,str): continue
    slate=[]
    for tkn in imps.split():
        p=tkn.split("-")
        if len(p)==2: slate.append((pfx(p[0]), int(p[1])))
    imp_slate[iid]=slate

def _auc(s,lb):
    p=lb==1;n=lb==0;np_,nn=p.sum(),n.sum()
    if np_==0 or nn==0: return None
    o=np.argsort(s);r=np.empty_like(o,float);r[o]=np.arange(1,len(s)+1)
    return float((r[p].sum()-np_*(np_+1)/2)/(np_*nn))

def per_impression_auc(cache):
    """For each impression the cache covers, compute AUC over its slate. Returns dict iid->auc."""
    # group cache keys by impression
    by_imp={}
    for (iid,aid),sc in cache.items():
        by_imp.setdefault(iid,{})[aid]=sc
    out={}
    for iid,scores in by_imp.items():
        slate=imp_slate.get(iid)
        if not slate: continue
        s=[]; lb=[]
        for aid,lab in slate:
            if aid in scores:
                s.append(scores[aid]); lb.append(lab)
        if len(s)<2 or sum(lb)==0: continue
        a=_auc(np.array(s),np.array(lb))
        if a is not None: out[iid]=a
    return out

auc_base = per_impression_auc(base)
auc_ta   = per_impression_auc(ta)
auc_cat  = per_impression_auc(cat)
auc_fresh= per_impression_auc(fresh)
print("per-impression AUC computed:",
      "base",len(auc_base),"ta",len(auc_ta),"cat",len(auc_cat),"fresh",len(auc_fresh))


In [ ]:
# ---- GBDT per-impression AUC keyed by impression_id (from npz iids_dv) -> PAIRABLE ----
import lightgbm as lgb
d=np.load(dl("mind_large_features.npz"),allow_pickle=True)
Xdv,ydv,gdv=d["Xdv"].astype(np.float32),d["ydv"],[int(g) for g in d["gdv"]]
iids_dv=list(d["iids_dv"]) if "iids_dv" in d.files else None
lgbm=lgb.Booster(model_file=dl("mind_lgbm_reranker.txt"))
sc=lgbm.predict(Xdv)
auc_gbdt={}
pos=0
for gi,iid in zip(gdv, (iids_dv if iids_dv is not None else range(len(gdv)))):
    a=_auc(sc[pos:pos+gi],ydv[pos:pos+gi]); pos+=gi
    if a is not None and iids_dv is not None: auc_gbdt[iid]=a
if iids_dv is None:
    print("WARNING: npz has no iids_dv -> re-run A2_01 (updated) to enable paired GBDT CI.")
    # fall back to unpaired list
    auc_gbdt_list=[]; pos=0
    for gi in gdv:
        a=_auc(sc[pos:pos+gi],ydv[pos:pos+gi]); pos+=gi
        if a is not None: auc_gbdt_list.append(a)
    auc_gbdt_list=np.array(auc_gbdt_list)
print("GBDT per-impression AUC:", len(auc_gbdt) if iids_dv is not None else len(auc_gbdt_list))


In [ ]:
# ---- PAIRED bootstrap 95% CI: improvement vs baseline, per-impression AUC delta ----
def paired_ci(auc_a, auc_b, nb=2000):
    """auc_a, auc_b: dicts iid->auc. Pair on shared iids; bootstrap mean(a-b)."""
    shared=sorted(set(auc_a)&set(auc_b))
    deltas=np.array([auc_a[i]-auc_b[i] for i in shared])
    if len(deltas)==0: return None
    rng=np.random.default_rng(0)
    boots=np.array([rng.choice(deltas,len(deltas),replace=True).mean() for _ in range(nb)])
    return dict(n=len(deltas), mean=deltas.mean(),
                lo=np.percentile(boots,2.5), hi=np.percentile(boots,97.5),
                base_auc=np.mean([auc_b[i] for i in shared]),
                impr_auc=np.mean([auc_a[i] for i in shared]))

print("="*70)
print("Q3.4  PAIRED BOOTSTRAP 95% CI  (per-impression AUC delta vs NRMS baseline)")
print("="*70)

def show(name, res):
    if res is None: print(f"{name}: no shared impressions"); return
    sig = "SIGNIFICANT (excludes 0)" if (res['lo']>0 or res['hi']<0) else "not significant (spans 0)"
    print(f"\n{name}  (n={res['n']} paired impressions)")
    print(f"  baseline AUC {res['base_auc']:.4f} -> improved AUC {res['impr_auc']:.4f}")
    print(f"  mean delta {res['mean']:+.4f}   95% CI [{res['lo']:+.4f}, {res['hi']:+.4f}]   -> {sig}")

# Improvement 2 (neural): title+abstract vs baseline
show("IMPROVEMENT 2  NRMS title+abstract  vs  NRMS title-only", paired_ci(auc_ta, auc_base))

# Ablation: +category vs baseline (expected: small/positive but the KEY compare is vs title+abstract)
show("ABLATION  NRMS +category  vs  NRMS title-only", paired_ci(auc_cat, auc_base))
show("ABLATION  NRMS +category  vs  NRMS title+abstract (isolates category)", paired_ci(auc_cat, auc_ta))

# Ablation: +recency vs baseline and vs title+abstract (expected regression vs title+abstract)
show("ABLATION  NRMS +recency  vs  NRMS title+abstract (isolates recency)", paired_ci(auc_fresh, auc_ta))


In [ ]:
# ---- Improvement 1 (features): GBDT vs baseline ----
def paired_ci2(auc_a, auc_b, nb=2000):
    shared=sorted(set(auc_a)&set(auc_b))
    deltas=np.array([auc_a[i]-auc_b[i] for i in shared])
    if len(deltas)==0: return None
    rng=np.random.default_rng(0)
    boots=np.array([rng.choice(deltas,len(deltas),replace=True).mean() for _ in range(nb)])
    return dict(n=len(deltas),mean=deltas.mean(),lo=np.percentile(boots,2.5),hi=np.percentile(boots,97.5),
                base_auc=np.mean([auc_b[i] for i in shared]),impr_auc=np.mean([auc_a[i] for i in shared]))
print("="*70)
print("IMPROVEMENT 1  LightGBM (behavioural features)  vs  NRMS baseline")
print("="*70)
if 'auc_gbdt' in dir() and len(auc_gbdt)>0 and len(set(auc_gbdt)&set(auc_base))>0:
    r=paired_ci2(auc_gbdt, auc_base)
    sig="SIGNIFICANT (excludes 0)" if (r['lo']>0 or r['hi']<0) else "not significant"
    print(f"  PAIRED (n={r['n']}): baseline AUC {r['base_auc']:.4f} -> GBDT AUC {r['impr_auc']:.4f}")
    print(f"  mean delta {r['mean']:+.4f}   95% CI [{r['lo']:+.4f}, {r['hi']:+.4f}]   -> {sig}")
else:
    base_aucs=np.array(list(auc_base.values()))
    rng=np.random.default_rng(0)
    boots=np.array([rng.choice(auc_gbdt_list,len(auc_gbdt_list),replace=True).mean()
                    -rng.choice(base_aucs,len(base_aucs),replace=True).mean() for _ in range(2000)])
    m=auc_gbdt_list.mean()-base_aucs.mean(); lo,hi=np.percentile(boots,2.5),np.percentile(boots,97.5)
    sig="SIGNIFICANT (excludes 0)" if (lo>0 or hi<0) else "not significant"
    print(f"  UNPAIRED: GBDT AUC {auc_gbdt_list.mean():.4f} vs baseline {base_aucs.mean():.4f}")
    print(f"  mean delta {m:+.4f}   95% CI [{lo:+.4f}, {hi:+.4f}]   -> {sig}")
    print("  (re-run updated A2_01 to get iids_dv -> strictly PAIRED CI)")


In [ ]:
# ---- Q3 SUMMARY TABLE ----
print("="*70)
print("Q3 SUMMARY  (MIND dev)  -- baseline reproduced, then beaten, with CIs")
print("="*70)
rows=[
 ("NRMS title-only (BASELINE)", np.mean(list(auc_base.values()))),
 ("NRMS +recency (ablation)",   np.mean(list(auc_fresh.values()))),
 ("NRMS +category (ablation)",  np.mean(list(auc_cat.values()))),
 ("NRMS title+abstract (IMPROVEMENT 2)", np.mean(list(auc_ta.values()))),
 ("LightGBM features (IMPROVEMENT 1)",   (np.mean(list(auc_gbdt.values())) if ('auc_gbdt' in dir() and len(auc_gbdt)>0) else auc_gbdt_list.mean())),
]
for name,a in rows:
    print(f"  {name:40s} AUC {a:.4f}")
print("\nBoth improvements beat the baseline with 95% CIs excluding zero (above).")
print("Category = null (CI vs title+abstract spans 0); recency = regression.")
